# Lekcija 12 - Zmanjšanje zgodovine klepeta z agentovim delovnim zvezkom

Ta zvezek prikazuje, kako upravljati s kontekstom v dolgih pogovorih z uporabo Microsoft Agent Framework. Ko se pogovori daljšajo, se število žetonov povečuje — sčasoma presežejo kontekstno okno modela. To naslovimo z **vzorec povzetka konteksta** in **agentovim delovnim zvezkom** za trajni pomnilnik.

## Kaj se boste naučili:
1. **Zakaj je upravljanje konteksta pomembno**: Razumevanje omejitev žetonov in kontekstnih oken
2. **Agentii, ki upoštevajo kontekst**: Gradnja agentov, ki upravljajo svoj lasten kontekst pogovora
3. **Vzorec povzetka konteksta**: Uporaba orodij za povzema zgodovine pogovora
4. **Agentov delovni zvezek**: Trajni pomnilnik, ki preživi zmanjšanje konteksta

## Predpogoji:
- Nastavitev Azure OpenAI z konfiguriranimi okoljskimi spremenljivkami
- Razumevanje osnovnih konceptov agentov iz prejšnjih lekcij


## Nastavitev


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Azure AI Foundry client configured")

## Zakaj je upravljanje konteksta pomembno

Vsak LLM ima končno **povečavo konteksta** — največje število žetonov, ki jih lahko obdela v enem samem pozivu. Ko poteka večvrstični pogovor:

- **Število žetonov linearno raste** z vsakim uporabniškim sporočilom in odgovorom asistenta.
- **Stroški večinoma povzročajo pozivni žetoni**, ker se celotna zgodovina vsakič znova pošlje.
- Na koncu pogovor **preseže povečavo konteksta** in model bodisi skrajša vsebino bodisi javi napako.

### Strategije za upravljanje konteksta

| Strategija | Kako deluje | Kompromis |
|---|---|---|
| **Skrajšanje** | Odstrani najstarejša sporočila | Izguba zgodnjega konteksta |
| **Povzetek** | Stisne starejša sporočila v povzetek | Nekatere podrobnosti izgubljene, a ključne točke ohranjene |
| **Zvezek / Zunanji spomin** | Shrani ključna dejstva izven pogovora | Zahteva klice orodij, vendar preživi vsako zmanjšanje |

V tem zapisku združujemo **povzetke** z orodjem **zvezka**, da agent ohranja kontinuiteto tudi, ko je zgodovina pogovora strnjena.


## Ustvarjanje agenta, ki se zaveda konteksta


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulacija dolgega pogovora

Pojdimo skozi večstopenjski pogovor, da vidimo, kako se kontekst kopiči. Agent mora ohraniti ključne podrobnosti (priljubljenosti, proračun, datume potovanja) skozi različne korake in pokazati kontinuiteto.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Opazite, kako agent ohranja kontekst iz prejšnjih krogov — ve za Japonsko, sushi, templje, fotografijo, proračun 3000 $, samostojno potovanje in april kot časovno okno. V kratkem pogovoru to deluje dobro, vendar pa je pri daljšem pogovoru ponovna pošiljanja celotne zgodovine draga.

Nadaljujmo pogovor z več odzivi, da vidimo kopičenje konteksta:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Vzorec povzemanja konteksta

Ko se pogovor razvija, lahko uporabimo **orodje za povzemanje**, da strnemo zbrane podatke v jedrnat format. Agent pokliče to orodje, da zabeleži ključne preference, tako da tudi če starejša sporočila izginejo, je bistvena informacija ohranjena.

Ta vzorec je gradnik za bolj prefinjeno zmanjševanje zgodovine:
1. Agent prepozna ključne podatke iz pogovora
2. Pokliče orodje za povzemanje, da jih shrani
3. Starejša sporočila je mogoče varno odstraniti, saj povzetek zajame bistvo

Spodaj definiramo orodje `summarize_preferences`, ki ga lahko agent pokliče, da zabeleži jedrnat povzetek tega, kar se je naučil.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Povzetek

V tej lekciji ste se naučili upravljati kontekst v dolgotrajnih pogovorih agentov z uporabo Microsoft Agent Framework:

### Ključni pojmi
- **Kontekstna okna so omejena** — vsak token v zgodovini pogovora stane in se šteje v omejitev.
- **Orodja za povzemanje** omogočajo agentu, da zgošči nabrani kontekst v kompaktne povzetke, s čimer zmanjša uporabo tokenov ob ohranjanju bistvenih informacij.
- **Agentove zvezke** nudijo trajni zunanji pomnilnik, ki preživi vse zmanjšanje pogovora.

### Kaj ste zgradili
- **agenta, ki pozna kontekst**, in ohranja kontinuiteto skozi večvračalne pogovore
- **orodje za povzemanje** (`summarize_preferences`), ki zabeleži ključne podatke uporabnika v kompaktni obliki
- **večvračalen pogovor**, ki prikazuje zadrževanje konteksta in obvladovanje sprememb

### Praktične uporabe
- **Boti za pomoč uporabnikom**: Pomnijo preference skozi dolge podporne seje
- **Osebni asistenti**: Spremljajo tekoče projekte brez ponovnega razlaganja konteksta
- **Izobraževalni tutorji**: Ohranjajo napredek študentov skozi številne interakcije

### Naslednji koraki
- Implementirati orodje zvezka z datotečno trajnostjo
- Dodati samodejno krajšanje zgodovine po povzetju
- Združiti z vektorskimi zbirkami za semantično iskanje po pomnilniku
- Zgraditi agente, ki lahko nadaljujejo pogovore dneve kasneje s polnim kontekstom


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
